# WDBC Breast Cancer Classification

Notebook format is simple and direct.
Each model has its own section so the results are easy to compare.


## Setup And Data


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

try:
    import tensorflow as tf
    TF_OK = True
except Exception:
    TF_OK = False

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

features = ["radius", "texture", "perimeter", "area", "smoothness", "compactness", "concavity", "concave_points", "symmetry", "fractal_dimension"]
columns = ["id", "diagnosis"] + [f"{name}_{part}" for part in ("mean", "se", "worst") for name in features]

df = pd.read_csv(ROOT / "data" / "wdbc.data", names=columns)
print(df.head())

df["target"] = (df["diagnosis"] == "M").astype(int)
print(df["diagnosis"].value_counts())


In [ ]:
train, temp = train_test_split(df, test_size=0.4, stratify=df["target"], random_state=42)
valid, test = train_test_split(temp, test_size=0.5, stratify=temp["target"], random_state=42)

x_train_raw = train.drop(columns=["id", "diagnosis", "target"]).values
x_valid_raw = valid.drop(columns=["id", "diagnosis", "target"]).values
x_test_raw = test.drop(columns=["id", "diagnosis", "target"]).values
y_train = train["target"].values
y_valid = valid["target"].values
y_test = test["target"].values

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_raw)
x_valid = scaler.transform(x_valid_raw)
x_test = scaler.transform(x_test_raw)

summary_rows = []

# Compare models on the validation split only. Keep the test split untouched for the final check.
def show_result(name, y_true, y_pred, y_prob):
    summary_rows.append({
        "model": name,
        "accuracy": round(accuracy_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred), 4),
        "recall": round(recall_score(y_true, y_pred), 4),
        "f1": round(f1_score(y_true, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_true, y_prob), 4),
    })
    print(name)
    print("Validation ROC-AUC:", round(roc_auc_score(y_true, y_prob), 4))
    print(classification_report(y_true, y_pred))


## Distance Based Model

### K-Nearest Neighbours (kNN)


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(x_train, y_train)
y_pred = knn.predict(x_valid)
y_prob = knn.predict_proba(x_valid)[:, 1]
show_result("kNN", y_valid, y_pred, y_prob)


## Probabilistic Model

### Naive Bayes


In [ ]:
gnb = GaussianNB()
gnb.fit(x_train, y_train)
y_pred = gnb.predict(x_valid)
y_prob = gnb.predict_proba(x_valid)[:, 1]
show_result("Naive Bayes", y_valid, y_pred, y_prob)


## Linear Model

### Logistic Regression


In [ ]:
logreg = LogisticRegression(max_iter=3000)
logreg.fit(x_train, y_train)
y_pred = logreg.predict(x_valid)
y_prob = logreg.predict_proba(x_valid)[:, 1]
show_result("Logistic Regression", y_valid, y_pred, y_prob)


## Margin Based Model

### Support Vector Machine (SVM)


In [ ]:
svm = SVC(probability=True)
svm.fit(x_train, y_train)
y_pred = svm.predict(x_valid)
y_prob = svm.predict_proba(x_valid)[:, 1]
show_result("SVM", y_valid, y_pred, y_prob)


## Tree Based Models

### Decision Tree


In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(x_train, y_train)
y_pred = tree.predict(x_valid)
y_prob = tree.predict_proba(x_valid)[:, 1]
show_result("Decision Tree", y_valid, y_pred, y_prob)


### Random Forest


In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(x_train, y_train)
y_pred = rf.predict(x_valid)
y_prob = rf.predict_proba(x_valid)[:, 1]
show_result("Random Forest", y_valid, y_pred, y_prob)


### Gradient Boosting


In [ ]:
gb = GradientBoostingClassifier(random_state=42)
gb.fit(x_train, y_train)
y_pred = gb.predict(x_valid)
y_prob = gb.predict_proba(x_valid)[:, 1]
show_result("Gradient Boosting", y_valid, y_pred, y_prob)


## Neural Networks

### sklearn MLPClassifier


In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, random_state=42)
mlp.fit(x_train, y_train)
y_pred = mlp.predict(x_valid)
y_prob = mlp.predict_proba(x_valid)[:, 1]
show_result("MLPClassifier", y_valid, y_pred, y_prob)


### TensorFlow


In [ ]:
if TF_OK:
    tf.random.set_seed(42)
    nn = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(x_train.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    nn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
    nn.fit(x_train, y_train, epochs=20, batch_size=32, validation_data=(x_valid, y_valid), verbose=0)
    y_prob = nn.predict(x_valid, verbose=0).ravel()
    y_pred = (y_prob > 0.5).astype(int)
    show_result("TensorFlow NN", y_valid, y_pred, y_prob)
else:
    print("TensorFlow is not installed in this environment yet.")


## Validation Ranking

Keep the full validation leaderboard instead of forcing a single winner. Models are ordered by validation ROC-AUC, then accuracy, then F1.


In [ ]:
summary = pd.DataFrame(summary_rows).sort_values(["roc_auc", "accuracy", "f1"], ascending=False).reset_index(drop=True)
top_score = summary.loc[0, "roc_auc"]
top_models = summary.loc[summary["roc_auc"] == top_score, "model"].tolist()

print("Validation ranking (ROC-AUC, accuracy, F1)")
print("Top validation model(s):", ", ".join(top_models))
summary
